In [1]:
# 1.1 Import Library
import pandas as pd
import re

In [2]:
# 1.2 Load Data Hasil Scraping
df = pd.read_csv("datascraping/master_dataset_aspirasi_dpr.csv")

print("Data berhasil dimuat.")
print("Jumlah data awal:", len(df))

df.head()

Data berhasil dimuat.
Jumlah data awal: 16379


,timestamp,username,text,url_pencarian,tahun
0,2016-12-30T06:37:56.000Z,@majudepan578,ADIL loh utk yg punya kebijakan publik & negar...,https://x.com/search?q=(%22DPR%20RI%22%20OR%20...,2016
1,2016-12-30T06:30:36.000Z,NaN,"Tertibkan Media Online, DPR: Pemerintah Jangan...",https://x.com/search?q=(%22DPR%20RI%22%20OR%20...,2016
2,2016-12-30T04:48:35.000Z,@jiggerbunny,@Portal_Kemlu_RI @DPR_RI @jokowi harus dievalu...,https://x.com/search?q=(%22DPR%20RI%22%20OR%20...,2016
3,2016-12-30T04:21:40.000Z,@marunduri_tri,"jangan ngambang, aturan logis apa undang-undan...",https://x.com/search?q=(%22DPR%20RI%22%20OR%20...,2016
4,2016-12-30T02:36:13.000Z,NaN,6. Kebebasan bersuara & berpendapat memang dij...,https://x.com/search?q=(%22DPR%20RI%22%20OR%20...,2016


In [3]:
# 1.3 Pemeriksaan Struktur Dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16379 entries, 0 to 16378
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   timestamp      16379 non-null  object
 1   username       15921 non-null  object
 2   text           16379 non-null  object
 3   url_pencarian  16379 non-null  object
 4   tahun          16379 non-null  int64 
dtypes: int64(1), object(4)
memory usage: 639.9+ KB


In [4]:
# 1.4 Attribute Selection (Pemilihan Atribut)
df_selected = df[['timestamp', 'text']].copy()

print("Atribut yang digunakan: timestamp dan text")
df_selected.head()

Atribut yang digunakan: timestamp dan text


,timestamp,text
0,2016-12-30T06:37:56.000Z,ADIL loh utk yg punya kebijakan publik & negar...
1,2016-12-30T06:30:36.000Z,"Tertibkan Media Online, DPR: Pemerintah Jangan..."
2,2016-12-30T04:48:35.000Z,@Portal_Kemlu_RI @DPR_RI @jokowi harus dievalu...
3,2016-12-30T04:21:40.000Z,"jangan ngambang, aturan logis apa undang-undan..."
4,2016-12-30T02:36:13.000Z,6. Kebebasan bersuara & berpendapat memang dij...


In [5]:
# 1.5 Record Selection (Filtering Dasar)

# Menghapus nilai text yang NaN
df_selected = df_selected[df_selected['text'].notna()]

# Menghapus text kosong atau hanya spasi
df_selected = df_selected[df_selected['text'].str.strip() != ""]

# Reset index setelah filtering
df_selected = df_selected.reset_index(drop=True)

print("Jumlah data setelah record selection:", len(df_selected))

Jumlah data setelah record selection: 16379


In [6]:
# 1.6 Relevance Filtering (LENGKAP untuk DPR RI)
import re

def is_relevant_tweet(text):
    """
    Filter tweet yang mengandung opini/kritik/dukung terhadap DPR RI.
    Keywords disesuaikan dengan konteks politik Indonesia di Twitter.
    """
    if not isinstance(text, str):
        return False
    
    text_lower = text.lower()
    
    # --- Hapus URL dulu untuk cek konten sebenarnya ---
    text_without_url = re.sub(r'http[s]?://\S+', '', text_lower).strip()
    
    # Jika setelah hapus URL teksnya terlalu pendek → tidak relevan
    if len(text_without_url.split()) < 3:
        return False
    
    # KATEGORI 1: SIKAP/EMOSI (Langsung menunjukkan opini)
    emotion_keywords = [
        # Positif
        'dukung', 'dukunglah', 'dukungin', 'dukungannya', 'dukung2',
        'setuju', 'setujukah', 'persetujuan', 'setuju banget',
        'senang', 'senangnya', 'bahagia', 'bersyukur', 'syukur',
        'bangga', 'salut', 'hormat', 'terima kasih', 'makasih',
        'puas', 'puas banget', 'legowo', 'ikhlas',
        'mantap', 'keren', 'hebat', 'bagus', 'baik', 'luar biasa',
        'apresiasi', 'mengapresiasi', 'terapresiasi',
        
        # Negatif
        'kritik', 'kritikan', 'mengkritik', 'dikritik', 'kritikannya',
        'kecewa', 'kecewanya', 'mengecewakan', 'kecewa banget',
        'marah', 'marahnya', 'kemarahan', 'emosi', 'geram',
        'kesal', 'gemas', 'dongkol', 'jengkel', 'sebel',
        'tidak puas', 'tidak senang', 'tidak setuju', 'tidak bangga',
        'buruk', 'jelek', 'parah', 'kacau', 'berantakan', 'rusak',
        'zonk', 'kecewa berat', 'sangat kecewa',
        
        # Netral tapi evaluatif
        'harap', 'harapan', 'diharapkan', 'berharap', 'harapkan',
        'tolak', 'menolak', 'penolakan', 'ditolak', 'menolaknya',
        'pro', 'kontra', 'dukung', 'lawan',
        'ragu', 'ragu-ragu', 'tidak yakin', 'pertanyaan',
    ]
    
    # KATEGORI 2: KINERJA DAN TANGGUNG JAWAB DPR
    performance_keywords = [
        'kerja', 'bekerja', 'kinerja', 'berkinerja', 'pekerjaan',
        'malas', 'aktif', 'pasif', 'rajin', 'sibuk',
        'tanggung jawab', 'bertanggung jawab', 'abai', 'acuh',
        'peduli', 'tidak peduli', 'mengabaikan', 'abaikan',
        'janji', 'menepati janji', 'ingkar janji', 'bohong', 'dusta',
        'komitmen', 'berkomitmen', 'tidak komitmen',
        'fungsi', 'berfungsi', 'tidak berfungsi', 'macet',
        'pengawasan', 'mengawasi', 'diawasi', 'kontrol',
    ]
    
    # KATEGORI 3: LEGISLASI DAN KEBIJAKAN
    legislation_keywords = [
        'ruu', 'uu', 'undang-undang', 'perundang-undangan',
        'peraturan', 'regulasi', 'kebijakan', 'policy',
        'sah', 'sahkan', 'disahkan', 'pengesahan', 'mensahkan',
        'batal', 'batalkan', 'dibatalkan', 'pembatalan',
        'revisi', 'merevisi', 'direvisi', 'amandemen', 'mengamandemen',
        'prolegnas', 'prolegda', 'legislasi', 'legislatif',
        'bahas', 'membahas', 'dibahas', 'pembahasan',
        'godok', 'menggodok', 'digodok', 'penggodokan',
        'pasal', 'ayat', 'poin', 'poin-poin', 'butir',
        'fraksi', 'koalisi', 'partai', 'politisi',
    ]
    
    # KATEGORI 4: PELAYANAN PUBLIK DAN RAKYAT
    public_service_keywords = [
        'rakyat', 'masyarakat', 'publik', 'konstituen', 'warga',
        'aspirasi', 'aspirasinya', 'menampung aspirasi',
        'suara', 'suaranya', 'perwakilan', 'mewakili',
        'melayani', 'pelayanan', 'dilayani', 'tidak dilayani',
        'mengabdi', 'pengabdian', 'abdikan',
        'kepentingan', 'kepentingan rakyat', 'kepentingan umum',
        'kesejahteraan', 'sejahtera', 'kemakmuran',
        'hak', 'hak rakyat', 'hak konstituen', 'kewajiban',
    ]
    
    # KATEGORI 5: KORUPSI, SKANDAL DAN ETIKA
    corruption_keywords = [
        'korupsi', 'koruptor', 'koruptif', 'tikus berdasi',
        'suap', 'menyuap', 'disuap', 'penyuapan', 'gratifikasi',
        'ott', 'operasi tangkap tangan', 'tangkap', 'ditangkap',
        'skandal', 'kasus', 'tersangka', 'terdakwa', 'terpidana',
        'etika', 'etik', 'pelanggaran etika', 'kode etik',
        'sanksi', 'hukuman', 'pidana', 'denda',
        'uang rakyat', 'anggaran', 'APBN', 'APBD', 'korup',
        'mewah', 'hedon', 'foya-foya', 'boros', 'pamer',
    ]
    
    # KATEGORI 6: PERTANYAAN DAN KERAGUAN (Indikator Opini)
    question_keywords = [
        'kenapa', 'mengapa', 'bagaimana', 'kapan', 'berapa',
        'mana', 'di mana', 'kemana', 'darimana', 'siapa',
        'apakah', 'apa', 'siapa', 'kenapa tidak', 'mengapa tidak',
        'kapan sih', 'kapan ya', 'sampai kapan',
        'kok bisa', 'kok tidak', 'kenapa sih',
        'masa', 'masa sih', 'masa iya', 'beneran', 'serius',
    ]
    
    # KATEGORI 7: PERBANDINGAN DAN EVALUASI
    comparison_keywords = [
        'lebih baik', 'lebih buruk', 'lebih bagus', 'lebih jelek',
        'dibanding', 'dibandingkan', 'daripada', 'ketimbang',
        'seharusnya', 'harusnya', 'selayaknya', 'sepatutnya',
        'sebaiknya', 'alangkah baiknya', 'akan lebih baik',
        'kurang', 'terlalu', 'cukup', 'sangat', 'amat', 'banget',
        'paling', 'ter-', 'terbaik', 'terburuk', 'terparah',
    ]
    
    # KATEGORI 8: NEGATION PLUS INTENSIFIER (Bahasa Indonesia Informal)
    negation_keywords = [
        'tidak', 'tak', 'nggak', 'gak', 'ngga', 'gk',
        'bukan', 'jangan', 'tanpa', 'belum', 'no', 'never',
        'enggak', 'ga', 'ndak', 'tak', 'botol',  # slang negation
    ]
    
    intensifier_keywords = [
        'sangat', 'amat', 'sekali', 'banget', 'betul', 'benar',
        'teramat', 'terlalu', 'cukup', 'lumayan', 'agak',
        'sedikit', 'kurang', 'hampir', 'nyaris', 'hanya',
        'cuma', 'sekadar', 'mungkin', 'sepertinya', 'rasanya',
        'parah', 'gila', 'negeri', 'banget sih', 'kok',
    ]
    
    # GABUNGKAN SEMUA KEYWORDS
    all_opinion_keywords = (
        emotion_keywords + 
        performance_keywords + 
        legislation_keywords + 
        public_service_keywords + 
        corruption_keywords + 
        question_keywords + 
        comparison_keywords + 
        negation_keywords + 
        intensifier_keywords
    )
    
    has_opinion = any(kw in text_without_url for kw in all_opinion_keywords)
    
    # KRITERIA EKSKLUSI: Hanya informasi formal
    info_patterns = [
        r'^rt\s',                    # Retweet tanpa komentar
        r'\bcc\b',                   # Carbon copy (informasi)
        r'agenda\s+dpr',             # Agenda formal
        r'jadwal\s+dpr',
        r'rapat\s+paripurna',        # Rapat formal
        r'rapat\s+pansus',
        r'rapat\s+panja',
        r'rapat\s+dengar\s+pendapat',
        r'rdp\s+dpr',
        r'hasil\s+rapat',
        r'publikasi\s+dpr',
        r'berita\s+dpr',
        r'siaran\s+pers',
        r'live\s+streaming',
        r'link\s+berita',
        r'klik\s+link',
        r'baca\s+selengkapnya',
        r'foto\s+kegiatan',
        r'dokumentasi\s+dpr',
        r'unduh\s+',
        r'download\s+',
    ]
    
    is_info_only = any(re.search(p, text_without_url) for p in info_patterns)
    
    # KEPUTUSAN
    if has_opinion:
        return True
    if is_info_only:
        return False
    
    # Fallback: jika ambigu, pertahankan (nanti bisa difilter manual)
    return len(text_without_url.split()) >= 5


# EKSEKUSI FILTER
print("RELEVANCE FILTERING (DPR RI Opinion Detection)")
print(f"Data sebelum filter: {len(df_selected):,} tweet")

df_selected = df_selected[
    df_selected['text'].apply(is_relevant_tweet)
].reset_index(drop=True)

print(f"Data setelah filter: {len(df_selected):,} tweet")
print(f"Data dihapus: {16379 - len(df_selected):,} tweet ({(16379 - len(df_selected))/16379*100:.2f}%)")

RELEVANCE FILTERING (DPR RI Opinion Detection)
Data sebelum filter: 16,379 tweet
Data setelah filter: 16,205 tweet
Data dihapus: 174 tweet (1.06%)


In [ ]:
# 1.7 Verifikasi Struktur Setelah Selection
df_selected.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16205 entries, 0 to 16204
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   timestamp  16205 non-null  object
 1   text       16205 non-null  object
dtypes: object(2)
memory usage: 253.3+ KB


In [ ]:
# 1.8 Penyimpanan Dataset Hasil Selection
df_selected.to_csv("dataselection/dataset_selected.csv", index=False)

print("Dataset selection berhasil disimpan.")

Dataset selection berhasil disimpan.
